In [13]:
import pandas as pd
import os
import csv
import re
import ftfy
import unicodedata
from collections import Counter
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer
from tokenizers.processors import TemplateProcessing
from tokenizers import Tokenizer
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
from sklearn.model_selection import train_test_split

torch.manual_seed(42)

In [2]:
data = pd.read_csv(r'..\data\processed\final_cleaned_jokes.csv')
data = data[data.joke_cleaned.notnull()]

In [3]:
tokenizer = Tokenizer.from_file(
    r'..\data\processed\tokenizer.json'
)

In [4]:
data

,stable_id,joke,topic,joke_cleaned,word_count
0,0000066daaacde54c609a69be4f00c3336480137,Whoever said imitation is the sincerest form o...,"imitation, flattery, minute",whoever said imitation is the sincerest form o...,23
1,00000a79878b3729adc0e47a34dbf5d5484214c0,"You're so fat, they oughta call your dick gary...","oldman, dick, gary","you're so fat , they oughta call your dick gar...",20
2,000054bc76448f9b302e6266a72324ecb81d4083,I couldn't sleep last night. I was wondering w...,"night, sun",i couldn't sleep last night . i was wondering ...,18
3,0000b76ab05f5d5cf9952d109c5ce7b143ae016f,What's 11 & 2? The Cowboys,cowboys,what's 11 2 ? the cowboys,6
4,00015c7571451cc5eb99c24dd7bc46a5ce46b9a1,I just got done apologizing to my barbershop q...,"barbershop, quartet, bucket",i just got done apologizing to my barbershop q...,34
...,...,...,...,...,...
726782,ffff42e7a4d6a0ba5d6d548400c2786eca9609c4,There were three dinosaurs who found a magic l...,"dinosaur, shower, genie",there were three dinosaurs who found a magic l...,70
726783,ffff4cf4bb9b9e63636efbb0692d20ea00257b64,What was the hardest part about Michael Jackso...,"michael, jackson, autograph",what was the hardest part about michael jackso...,18
726784,ffff621a689a09dd38aef56fc93b58fba9efd948,My dick is called life. Life is hard,"life, dick",my dick is called life . life is hard,9
726785,ffff6e0d16c8de3d9c1c9a1100747ab55f8d6e5f,What was Jeffrey Epstein humming before dying?...,"republic, jeffrey, epstein",what was jeffrey epstein humming before dying ...,17


In [5]:
encoded = tokenizer.encode(f"tell me a joke about {data.topic[0]}",data.joke_cleaned[0])
print(encoded.tokens)
print(encoded.ids)
print(tokenizer.decode(encoded.ids, skip_special_tokens=False))

['[S]', 'Ġtell', 'Ġme', 'Ġa', 'Ġjoke', 'Ġabout', 'Ġimitation', ',', 'Ġflattery', ',', 'Ġminute', '[JOKE]', 'Ġwhoever', 'Ġsaid', 'Ġimitation', 'Ġis', 'Ġthe', 'Ġsincerest', 'Ġform', 'Ġof', 'Ġflattery', 'Ġhasn', "'t", 'Ġhad', 'Ġa', 'Ġ7', 'yo', 'Ġmimicking', 'Ġtheir', 'Ġevery', 'Ġword', 'Ġfor', 'Ġthe', 'Ġlast', 'Ġ10', 'Ġminutes', 'Ġ.', '[/S]']
[0, 373, 140, 56, 403, 249, 22485, 12, 18861, 12, 2107, 6, 4522, 234, 22485, 127, 61, 23633, 2996, 110, 18861, 3060, 148, 303, 56, 967, 4546, 49147, 387, 365, 793, 157, 61, 522, 782, 1021, 62, 1]
[S] tell me a joke about imitation, flattery, minute[JOKE] whoever said imitation is the sincerest form of flattery hasn't had a 7yo mimicking their every word for the last 10 minutes .[/S]


In [6]:
sequences = [f"tell me a joke about {t} [JOKE] {j}" 
             for t,j in zip(data.topic, data.joke_cleaned)]

In [7]:
tokenized = tokenizer.encode_batch(sequences)

In [8]:
tokenized_ids = [x.ids for x in tokenized]

In [9]:
tokenized_ids = torch.tensor(tokenized_ids)

In [10]:
tokenized_ids.shape

torch.Size([726786, 256])

In [11]:
data_len = len(tokenized_ids)
train_data = tokenized_ids[:int(data_len*.9)]
test_data  = tokenized_ids[int(data_len*.9):]

In [14]:
train_data

tensor([[  0, 373, 140,  ...,   2,   2,   2],
        [  0, 373, 140,  ...,   2,   2,   2],
        [  0, 373, 140,  ...,   2,   2,   2],
        ...,
        [  0, 373, 140,  ...,   2,   2,   2],
        [  0, 373, 140,  ...,   2,   2,   2],
        [  0, 373, 140,  ...,   2,   2,   2]])

In [18]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, n_emb):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_emb)
        self.position_embedding = nn.Embedding(block_size, n_emb)
        
        self.lm_head = nn.Linear(n_emb, vocab_size)

    def forward(self, idx, targets=None):

        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    

In [17]:
torch.randint(256,(8,))

tensor([210, 214,  74, 202,  87, 116,  99, 103])